In [8]:
import pandas as pd

# 1. Load the CSV file using a two-level header
# The dataset exported from yfinance contains a multi-level header
df = pd.read_csv("/Users/aaron/Desktop/fyp/ts-node-esm-template/knowledge/csv/US_Sp500_history.csv", header=[0, 1])

# 2. Flatten the multi-level column names into a single level
df.columns = ['date', 'close', 'high', 'low', 'open', 'volume']

# 3. Remove the first invalid row that contains "Date" and NaN values
df = df[df['date'] != 'Date'].copy()

# 4. Convert columns to appropriate data types
df['date'] = pd.to_datetime(df['date'])

df['open'] = pd.to_numeric(df['open'], errors='coerce')
df['high'] = pd.to_numeric(df['high'], errors='coerce')
df['low'] = pd.to_numeric(df['low'], errors='coerce')
df['close'] = pd.to_numeric(df['close'], errors='coerce')
df['volume'] = pd.to_numeric(df['volume'], errors='coerce')

# 5. Remove rows with missing values
df = df.dropna()

# 6. Sort the dataset by date
df = df.sort_values('date').reset_index(drop=True)

# 7. Add a column to indicate the market
df['market'] = 'US'

# 8. Display the cleaned dataset
print("First rows of the cleaned dataset:")
print(df.head())

print("\nDataset information:")
print(df.info())

First rows of the cleaned dataset:
        date        close         high          low         open  \
0 2020-01-02  3257.850098  3258.139893  3235.530029  3244.669922   
1 2020-01-03  3234.850098  3246.149902  3222.340088  3226.360107   
2 2020-01-06  3246.280029  3246.840088  3214.639893  3217.550049   
3 2020-01-07  3237.179932  3244.909912  3232.429932  3241.860107   
4 2020-01-08  3253.050049  3267.070068  3236.669922  3238.590088   

         volume market  
0  3.459930e+09     US  
1  3.484700e+09     US  
2  3.702460e+09     US  
3  3.435910e+09     US  
4  3.726840e+09     US  

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1507 entries, 0 to 1506
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    1507 non-null   datetime64[ns]
 1   close   1507 non-null   float64       
 2   high    1507 non-null   float64       
 3   low     1507 non-null   float64       
 4   open    1

In [9]:
import numpy as np

# Daily return
df["return_1d"] = df["close"].pct_change()

# 5 day return
df["return_5d"] = df["close"].pct_change(5)

# 10 day return
df["return_10d"] = df["close"].pct_change(10)

# Moving averages
df["MA_5"] = df["close"].rolling(5).mean()
df["MA_20"] = df["close"].rolling(20).mean()

# MA gap
df["MA_gap"] = df["MA_5"] - df["MA_20"]

# Volatility (10 day std)
df["volatility_10"] = df["return_1d"].rolling(10).std()

# Volume change
df["volume_change"] = df["volume"].pct_change()

# RSI
delta = df["close"].diff()
gain = (delta.where(delta > 0, 0)).rolling(14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(14).mean()

rs = gain / loss
df["RSI_14"] = 100 - (100 / (1 + rs))

# MACD
ema12 = df["close"].ewm(span=12, adjust=False).mean()
ema26 = df["close"].ewm(span=26, adjust=False).mean()

df["MACD"] = ema12 - ema26

# Drop early NaN rows
df = df.dropna().reset_index(drop=True)

print(df.head())

        date        close         high          low         open  \
0 2020-01-30  3283.659912  3285.909912  3242.800049  3256.449951   
1 2020-01-31  3225.520020  3282.330078  3214.679932  3282.330078   
2 2020-02-03  3248.919922  3268.439941  3235.659912  3235.659912   
3 2020-02-04  3297.590088  3306.919922  3280.610107  3280.610107   
4 2020-02-05  3334.689941  3337.580078  3313.750000  3324.909912   

         volume market  return_1d  return_5d  return_10d         MA_5  \
0  3.790350e+09     US   0.003134  -0.012593   -0.001712  3274.479932   
1  4.529700e+09     US  -0.017706  -0.021226   -0.027523  3260.489941   
2  3.760460e+09     US   0.007255   0.001631   -0.024237  3261.547949   
3  3.996900e+09     US   0.014980   0.006517   -0.006986  3265.817969   
4  4.121480e+09     US   0.011251   0.018724    0.003896  3278.075977   

         MA_20     MA_gap  volatility_10  volume_change     RSI_14      MACD  
0  3280.837000  -6.357068       0.007703       0.052802  51.993049  5.976

In [10]:
import pandas as pd

def clean_market_data(file_path, market_name):
    
    # 1. Read CSV with two-level header
    df = pd.read_csv(file_path, header=[0,1])
    
    # 2. Flatten column names
    df.columns = ['date','close','high','low','open','volume']
    
    # 3. Remove invalid first row
    df = df[df['date'] != 'Date'].copy()
    
    # 4. Convert data types
    df['date'] = pd.to_datetime(df['date'])
    
    df['open'] = pd.to_numeric(df['open'], errors='coerce')
    df['high'] = pd.to_numeric(df['high'], errors='coerce')
    df['low'] = pd.to_numeric(df['low'], errors='coerce')
    df['close'] = pd.to_numeric(df['close'], errors='coerce')
    df['volume'] = pd.to_numeric(df['volume'], errors='coerce')
    
    # 5. Drop missing values
    df = df.dropna()
    
    # 6. Sort by date
    df = df.sort_values('date').reset_index(drop=True)
    
    # 7. Add market label
    df['market'] = market_name
    
    return df

In [11]:
df_us = clean_market_data("/Users/aaron/Desktop/fyp/ts-node-esm-template/knowledge/csv/US_Sp500_history.csv", "US")
df_hk = clean_market_data("/Users/aaron/Desktop/fyp/ts-node-esm-template/knowledge/csv/HK_HSI_history.csv", "HK")
df_sg = clean_market_data("/Users/aaron/Desktop/fyp/ts-node-esm-template/knowledge/csv/SG_STI_history.csv", "SG")
df_uk = clean_market_data("/Users/aaron/Desktop/fyp/ts-node-esm-template/knowledge/csv/UK_FTSE100_history.csv", "UK")
df_cn = clean_market_data("/Users/aaron/Desktop/fyp/ts-node-esm-template/knowledge/csv/CN_SSE_history.csv", "CN")

In [12]:
print(df_hk.head())
print(df_sg.head())
print(df_uk.head())
print(df_cn.head())

        date         close          high           low          open  \
0 2020-01-02  28543.519531  28543.519531  28245.970703  28249.369141   
1 2020-01-03  28451.500000  28883.300781  28428.169922  28828.359375   
2 2020-01-06  28226.189453  28367.869141  28054.289062  28326.500000   
3 2020-01-07  28322.060547  28473.080078  28264.070312  28352.679688   
4 2020-01-08  28087.919922  28198.609375  27857.730469  27999.580078   

         volume market  
0  1.262700e+09     HK  
1  1.797900e+09     HK  
2  1.793400e+09     HK  
3  1.302700e+09     HK  
4  1.709200e+09     HK  
        date        close         high          low         open       volume  \
0 2020-01-02  3252.000000  3256.090088  3220.669922  3230.479980  136381900.0   
1 2020-01-03  3238.820068  3268.729980  3227.629883  3260.270020  193650300.0   
2 2020-01-06  3218.860107  3226.110107  3210.679932  3221.560059  154063200.0   
3 2020-01-07  3247.860107  3247.860107  3233.500000  3236.229980  187657600.0   
4 2020-01-08

In [14]:
df_all = pd.concat([
    df_us,
    df_hk,
    df_sg,
    df_uk,
    df_cn
], ignore_index=True)

print(df_all.head())
print(df_all['market'].value_counts())
df_all.to_csv("global_market_raw.csv", index=False)

        date        close         high          low         open  \
0 2020-01-02  3257.850098  3258.139893  3235.530029  3244.669922   
1 2020-01-03  3234.850098  3246.149902  3222.340088  3226.360107   
2 2020-01-06  3246.280029  3246.840088  3214.639893  3217.550049   
3 2020-01-07  3237.179932  3244.909912  3232.429932  3241.860107   
4 2020-01-08  3253.050049  3267.070068  3236.669922  3238.590088   

         volume market  
0  3.459930e+09     US  
1  3.484700e+09     US  
2  3.702460e+09     US  
3  3.435910e+09     US  
4  3.726840e+09     US  
market
UK    1513
US    1507
SG    1507
HK    1475
CN    1454
Name: count, dtype: int64


In [15]:
import numpy as np

def add_features(df):

    df = df.sort_values("date")

    df["return_1d"] = df["close"].pct_change()
    df["return_5d"] = df["close"].pct_change(5)
    df["return_10d"] = df["close"].pct_change(10)

    df["MA_5"] = df["close"].rolling(5).mean()
    df["MA_20"] = df["close"].rolling(20).mean()

    df["MA_gap"] = df["MA_5"] - df["MA_20"]

    df["volatility_10"] = df["return_1d"].rolling(10).std()

    df["volume_change"] = df["volume"].pct_change()

    # RSI
    delta = df["close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()

    rs = gain / loss
    df["RSI_14"] = 100 - (100 / (1 + rs))

    # MACD
    ema12 = df["close"].ewm(span=12, adjust=False).mean()
    ema26 = df["close"].ewm(span=26, adjust=False).mean()

    df["MACD"] = ema12 - ema26

    return df


df_all = df_all.groupby("market", group_keys=False).apply(add_features)

df_all = df_all.dropna().reset_index(drop=True)

print(df_all.head())

        date        close         high          low         open  \
0 2020-01-30  3283.659912  3285.909912  3242.800049  3256.449951   
1 2020-01-31  3225.520020  3282.330078  3214.679932  3282.330078   
2 2020-02-03  3248.919922  3268.439941  3235.659912  3235.659912   
3 2020-02-04  3297.590088  3306.919922  3280.610107  3280.610107   
4 2020-02-05  3334.689941  3337.580078  3313.750000  3324.909912   

         volume market  return_1d  return_5d  return_10d         MA_5  \
0  3.790350e+09     US   0.003134  -0.012593   -0.001712  3274.479932   
1  4.529700e+09     US  -0.017706  -0.021226   -0.027523  3260.489941   
2  3.760460e+09     US   0.007255   0.001631   -0.024237  3261.547949   
3  3.996900e+09     US   0.014980   0.006517   -0.006986  3265.817969   
4  4.121480e+09     US   0.011251   0.018724    0.003896  3278.075977   

         MA_20     MA_gap  volatility_10  volume_change     RSI_14      MACD  
0  3280.837000  -6.357068       0.007703       0.052802  51.993049  5.976

/var/folders/vj/1_1wy3vs1bd3ly1kpy9bd33w0000gn/T/ipykernel_54714/1468627841.py:37: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_all = df_all.groupby("market", group_keys=False).apply(add_features)


In [16]:
df_all.to_csv("global_trading_dataset.csv", index=False)